In [1]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay


class ExptBuilder():
    def __init__(self):
        # Fail here, not one run into a 100-run scan. The old builder only ever
        # passed this path to `ar`, never opened it for writing, so a wrong path
        # stayed latent; write_experiment_to_file makes it fatal.
        self.__code_path__ = os.environ.get('code')
        if self.__code_path__ is None:
            raise EnvironmentError("the 'code' environment variable is not set.")

        self.__temp_exp_dir__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments",
                                             "HF_experiments", "feedback", "measurements")
        self.__temp_exp_path__ = os.path.join(self.__temp_exp_dir__,
                                              "apd_resolution_calibration_temp.py")
        if not os.path.isdir(self.__temp_exp_dir__):
            raise FileNotFoundError(
                f"temp experiment directory does not exist: {self.__temp_exp_dir__}")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & ar " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        return result.returncode

    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)

    def remove_experiment_file(self):
        # Unlike ry_scan_expt_builder, run_expt does NOT delete the temp file --
        # over a 100-run scan you want the failing source still on disk to look
        # at. Call this once the scan is done.
        if os.path.exists(self.__temp_exp_path__):
            os.remove(self.__temp_exp_path__)

    def apd_resolution_expt(self, amp_imaging):
        """apd_resolution_calibration.py with amp_imaging formatted in.

        amp_imaging stays a plain parameter rather than becoming an xvar, so
        every value gets its own run ID. That is the point of driving it from
        the builder: repeats of the same amp are separate runs, so run-to-run
        drift shows up as scatter between run IDs instead of being averaged
        away inside a single dataset.
        """
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.language import now_mu, at_mu, delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from numpy import int64

        class sigma_z(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=False,
                              camera_select=cameras.andor,
                              save_data=True,
                              imaging_type=img_types.DISPERSIVE)

                self.p.amp_imaging = {amp_imaging:.6g}
                self.p.t_pci_pulse = 5.e-6

                self.p.t_raman_pulse = 0.
                self.p.t_raman_pulse_offset = 127.e-9
                self.xvar('t_raman_pulse', self.p.t_raman_pi_pulse * np.linspace(0.,1.,3))

                self.p.t_tweezer_hold = 20.e-3
                self.p.t_tof = 20.e-6
                self.p.N_repeats = 25

                self.data.apd = self.data.add_data_container(2)

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare()

            @kernel
            def scan_kernel(self):
                self.integrator.init()

                self.set_imaging_detuning(frequency_detuned=self.p.frequency_detuned_hf_midpoint)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers()
                self.prep_raman()

                if self.p.t_raman_pulse > 0:
                    self.p.t_raman_pulse += self.p.t_raman_pulse_offset
                self.raman.pulse(self.p.t_raman_pulse)

                delay(50.e-6)

                self.integrated_imaging_pulse(self.data.apd, t=self.p.t_pci_pulse, idx=0)

                delay(self.p.t_tweezer_hold)

                self.tweezer.off()

                delay(100.e-3)

                self.integrated_imaging_pulse(self.data.apd, t=self.p.t_pci_pulse, idx=1)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                self.end(expt_filepath, restart_monitor=False)
        """)
        return script

In [2]:
eBuilder = ExptBuilder()

In [3]:
### amp_imaging scan
AMP_IMAGING_VALUES = np.linspace(0.1, 1.5, 9)
N_RUNS_PER_AMP = 10

# np.repeat (not np.tile) -> each amp is run N_RUNS_PER_AMP times back to back
# before moving on, rather than sweeping the whole list 5 times.
amp_imaging_list = np.repeat(AMP_IMAGING_VALUES, N_RUNS_PER_AMP)

np.random.default_rng().shuffle(amp_imaging_list)

print(f"{len(amp_imaging_list)} runs: {len(AMP_IMAGING_VALUES)} amp_imaging values "
      f"x {N_RUNS_PER_AMP} runs each")
print(f"amp_imaging {AMP_IMAGING_VALUES[0]:.6g} -> {AMP_IMAGING_VALUES[-1]:.6g}, "
      f"step {AMP_IMAGING_VALUES[1] - AMP_IMAGING_VALUES[0]:.6g}")

failures = []
for i, amp_imaging in enumerate(amp_imaging_list):
    print(f"--- run {i+1}/{len(amp_imaging_list)}: amp_imaging = {amp_imaging:.6g} ---")
    eBuilder.write_experiment_to_file(eBuilder.apd_resolution_expt(amp_imaging))
    returncode = eBuilder.run_expt()
    # 100 unattended runs is long enough that a silent failure partway through
    # would otherwise only turn up when the analysis came out short.
    if returncode != 0:
        failures.append((i, float(amp_imaging), returncode))

if failures:
    print(f"\n{len(failures)} of {len(amp_imaging_list)} runs returned nonzero:")
    for i, amp_imaging, returncode in failures:
        print(f"  run {i+1}: amp_imaging = {amp_imaging:.6g}, returncode {returncode}")
else:
    print(f"\nall {len(amp_imaging_list)} runs returned 0")

90 runs: 9 amp_imaging values x 10 runs each
amp_imaging 0.1 -> 1.5, step 0.175
--- run 1/90: amp_imaging = 0.1 ---
0  75 values of t_raman_pulse. 75 total shots.
Run ID: 76120

Sent: {'mask': 'spot', 'center': [1021, 821], 'phase': 0.38709699999999997, 'dimension': 30, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 30 um, phase = 0.38709699999999997 pi, x-center = 1021, y-center = 821

 Run ID: 76120
shot 1/75 done
shot 2/75 done
shot 3/75 done
shot 4/75 done
shot 5/75 done
shot 6/75 done
shot 7/75 done
shot 8/75 done
shot 9/75 done
shot 10/75 done
shot 11/75 done
shot 12/75 done
shot 13/75 done
shot 14/75 done
shot 15/75 done
shot 16/75 done
shot 17/75 done
shot 18/75 done
shot 19/75 done
shot 20/75 done
shot 21/75 done
shot 22/75 done
shot 23/75 done
shot 24/75 done
shot 25/75 done
shot 26/75 done
shot 27/75 done
shot 28/75 done
shot 29/75 done
shot 30/75 done
shot 31/75 done
shot 32/75 done
shot 33/75 done
shot 34/75 done
shot 35/75 done
shot 36/75 done

In [10]:
# Remove the temp experiment once the scan is done, so the ARTIQ experiment
# browser does not pick up a stale copy pinned to the last amp_imaging.
eBuilder.remove_experiment_file()